# Module 2 — Data Preprocessing
## Personalized TED Talks Recommendation System

**Notebook:** `02_Data_Preprocessing.ipynb`

### Objectives
1. Prepare the collected TED dataset for recommendation-system feature engineering.
2. Handle missing values, duplicate records, inconsistent data types, and text fields using documented rules.
3. Preserve useful talks even when transcripts are missing.
4. Create clean text and numerical metadata features for Module 3.
5. Validate and export a separate processed dataset.

### Skills
Pandas data handling, data validation, missing-value treatment, duplicate handling, text normalization, tokenization, categorical preparation, feature creation, and reproducible data pipelines.

**Important boundaries**
- Never overwrite the raw CSV.
- Keep original text columns; create separate cleaned columns.
- Do not invent user ratings or interaction history.
- Do not build TF-IDF/embeddings or train the recommender in this notebook. Those belong in Module 3/4.

## 1. Install and import dependencies

Install if required:

```bash
pip install pandas numpy
```

In [1]:
from pathlib import Path
import ast
import html
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

## 2. Configure project paths

The notebook expects the combined source file at `Scraped/Data/Raw/ted_talks.csv`. It automatically searches the current directory and its parent folders for the project root. If needed, set `PROJECT_ROOT` manually.

In [2]:
current = Path.cwd()
PROJECT_ROOT = None
RAW_PATH = None

for candidate in [current, *current.parents]:
    candidate_path = candidate / "Module_1_Data_Collection" / "Scraped" / "Data" / "Raw" / "ted_talks.csv"
    if not candidate_path.exists():
        candidate_path = candidate / "Scraped" / "Data" / "Raw" / "ted_talks.csv"
    if candidate_path.exists():
        PROJECT_ROOT = candidate
        RAW_PATH = candidate_path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find ted_talks.csv in Module_1_Data_Collection/Scraped/Data/Raw/ or Scraped/Data/Raw/"
    )

PROCESSED_DIR = RAW_PATH.parent.parent / "Processed"
OUTPUT_DIR = PROJECT_ROOT / "Module_2_Data_Exploration_Preprocessing" / "outputs"
REPORT_DIR = OUTPUT_DIR / "reports"
STATS_DIR = OUTPUT_DIR / "statistics"

for folder in [PROCESSED_DIR, REPORT_DIR, STATS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "ted_talks_preprocessed.csv"

print("Project root:", PROJECT_ROOT)
print("Input:", RAW_PATH)
print("Output:", OUTPUT_PATH)


Project root: c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation
Input: c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Scraped\Data\Raw\ted_talks.csv
Output: c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Scraped\Data\Processed\ted_talks_preprocessed.csv


## 3. Load and audit the raw dataset

We start from the original combined `ted_talks.csv`, not `ted_talks_clean.csv`. The existing clean file is a prior derivative and should not be treated as the raw source for this reproducible preprocessing pipeline.

In [3]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_PATH}")

raw_df = pd.read_csv(RAW_PATH, low_memory=False)
df = raw_df.copy(deep=True)

print("Raw shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
display(raw_df.head(3))
display(pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2)
}))

Raw shape: (2550, 20)
Columns: ['talk_id', 'title', 'main_speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'related_talks', 'ratings', 'views', 'comments', 'duration', 'published_date', 'event', 'url', 'source', 'film_date', 'languages', 'name', 'num_speaker']


,talk_id,title,main_speaker,speaker_occupation,description,tags,transcript,related_talks,ratings,views,comments,duration,published_date,event,url,source,film_date,languages,name,num_speaker
0,1,Do schools kill creativity?,Ken Robinson,Author/educator,Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (ra...,"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']","Good morning. How are you?(Laughter)It's been great, hasn't it? I've been blown away by the whole thing. In fact, I'...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/images/ted/172559_800x600.jpg', 'speaker': 'Ken Robinson', 'title': 'Bri...","[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, 'name': 'Beautiful', 'count': 4573}, {'id': 9, 'name': 'Ingen...",47227110,4553,1164,1151367060,TED2006,https://www.ted.com/talks/ken_robinson_says_schools_kill_creativity,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,60,Ken Robinson: Do schools kill creativity?,1
1,2,Averting the climate crisis,Al Gore,Climate advocate,"With the same humor and humanity he exuded in ""An Inconvenient Truth,"" Al Gore spells out 15 ways that individuals c...","['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainabili...","Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extr...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/images/ted/566c14767bd62c5ff760e483c5b16cd2753328cd_2880x1620.jpg', 'spe...","[{'id': 7, 'name': 'Funny', 'count': 544}, {'id': 3, 'name': 'Courageous', 'count': 139}, {'id': 2, 'name': 'Confusi...",3200520,265,977,1151367060,TED2006,https://www.ted.com/talks/al_gore_on_averting_climate_crisis,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,43,Al Gore: Averting the climate crisis,1
2,3,Simplicity sells,David Pogue,Technology columnist,"New York Times columnist David Pogue takes aim at technology’s worst interface-design offenders, and provides encour...","['computers', 'entertainment', 'interface design', 'media', 'music', 'performance', 'simplicity', 'software', 'techn...","(Music: ""The Sound of Silence,"" Simon & Garfunkel)Hello voice mail, my old friend.(Laughter)I've called for tech sup...","[{'id': 1725, 'hero': 'https://pe.tedcdn.com/images/ted/b7f415a054cc0a2bfdd90d0ad5a7f64cf060150d_1600x1200.jpg', 'sp...","[{'id': 7, 'name': 'Funny', 'count': 964}, {'id': 3, 'name': 'Courageous', 'count': 45}, {'id': 9, 'name': 'Ingeniou...",1636292,124,1286,1151367060,TED2006,https://www.ted.com/talks/david_pogue_says_simplicity_sells,TED_Public_Archive_CC_BY_NC_SA_4.0,1140739200,26,David Pogue: Simplicity sells,1


,dtype,missing,missing_percent
talk_id,int64,0,0.00
title,str,0,0.00
main_speaker,str,0,0.00
speaker_occupation,str,6,0.24
description,str,0,0.00
tags,str,0,0.00
transcript,str,86,3.37
related_talks,str,0,0.00
ratings,str,0,0.00
views,int64,0,0.00


## 4. Standardize column names and identify important fields

Column names are normalized to lowercase snake_case. The helper below recognizes common naming variations found in TED datasets. It does not rename or delete the source columns.

In [4]:
def normalize_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df.columns = [normalize_column_name(c) for c in df.columns]

def find_column(candidates, columns):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

COL = {
    "id": find_column(["talk_id", "id"], df.columns),
    "title": find_column(["title", "talk_title"], df.columns),
    "speaker": find_column(["speaker", "main_speaker", "speaker_name"], df.columns),
    "occupation": find_column(["speaker_occupation", "speaker_job"], df.columns),
    "description": find_column(["description", "talk_description"], df.columns),
    "tags": find_column(["tags", "topics", "tag"], df.columns),
    "transcript": find_column(["transcript", "transcripts"], df.columns),
    "duration": find_column(["duration", "duration_seconds"], df.columns),
    "published_date": find_column(["published_date", "date_published"], df.columns),
    "views": find_column(["views", "view_count"], df.columns),
    "comments": find_column(["comments", "comment_count"], df.columns),
    "url": find_column(["url", "talk_url"], df.columns),
}

display(pd.DataFrame.from_dict(COL, orient="index", columns=["detected_column"]))
missing_core = [key for key in ["title", "description", "tags", "transcript"] if COL[key] is None]
if COL["title"] is None:
    raise KeyError("A title column is required for TED talk preprocessing.")
if missing_core:
    print("Note: these optional content fields were not found:", missing_core)

,detected_column
id,talk_id
title,title
speaker,main_speaker
occupation,speaker_occupation
description,description
tags,tags
transcript,transcript
duration,duration
published_date,published_date
views,views


## 5. Measure missing data before handling it

This baseline is saved so the preprocessing report can show what changed. Empty strings and whitespace-only strings are treated as missing for analysis.

In [5]:
def missing_profile(frame):
    rows = []
    for col in frame.columns:
        nulls = int(frame[col].isna().sum())
        empty = int(frame[col].astype("string").str.strip().eq("").sum()) if frame[col].dtype == "object" else 0
        rows.append({
            "column": col,
            "null_count": nulls,
            "empty_string_count": empty,
            "missing_or_empty": nulls + empty,
            "missing_percent": round((nulls + empty) / len(frame) * 100, 2) if len(frame) else 0
        })
    return pd.DataFrame(rows).sort_values("missing_percent", ascending=False)

missing_before = missing_profile(df)
display(missing_before)

,column,null_count,empty_string_count,missing_or_empty,missing_percent
6,transcript,86,0,86,3.37
3,speaker_occupation,6,0,6,0.24
1,title,0,0,0,0.00
0,talk_id,0,0,0,0.00
2,main_speaker,0,0,0,0.00
4,description,0,0,0,0.00
5,tags,0,0,0,0.00
7,related_talks,0,0,0,0.00
8,ratings,0,0,0,0.00
9,views,0,0,0,0.00


## 6. Normalize empty values and remove confirmed duplicate records

We convert blank text cells to missing values. Exact duplicate rows are removed. If a talk identifier exists, duplicate IDs are resolved by keeping the first record after stable ordering. We report counts so this decision is auditable.

In [6]:
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

exact_duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

duplicate_id_removed = 0
if COL["id"] and COL["id"] in df.columns:
    duplicate_id_mask = df[COL["id"]].notna() & df[COL["id"]].duplicated(keep="first")
    duplicate_id_removed = int(duplicate_id_mask.sum())
    df = df.loc[~duplicate_id_mask].copy()

print("Exact duplicate rows removed:", exact_duplicates_before)
print("Duplicate talk IDs removed:", duplicate_id_removed)
print("Shape after duplicate handling:", df.shape)

Exact duplicate rows removed: 0
Duplicate talk IDs removed: 0
Shape after duplicate handling: (2550, 20)


## 7. Handle missing values using recommendation-aware rules

Rules:
- **Title:** a talk without a title is not useful for discovery, so remove it.
- **Speaker / occupation:** use `Unknown` labels rather than dropping the talk.
- **Description / transcript:** preserve missingness in source fields and use empty strings only in the derived cleaned-text fields.
- **Tags:** parse missing tags as an empty list; do not invent topics.
- **Numeric metadata:** convert invalid values to missing; do not replace them with fabricated values.
- Talks with missing transcripts remain in the dataset if other metadata can support content-based recommendation.

In [7]:
rows_before_missing_rules = len(df)

# A title is essential for a usable recommendation result.
title_col = COL["title"]
df[title_col] = df[title_col].astype("string").str.strip()
df = df[df[title_col].notna() & df[title_col].ne("")].copy()

for key in ["speaker", "occupation"]:
    col = COL[key]
    if col and col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].fillna("Unknown")
        df.loc[df[col].eq(""), col] = "Unknown"

# Keep original missing transcript/description values; text-cleaning functions handle NA safely.
for key in ["description", "transcript"]:
    col = COL[key]
    if col and col in df.columns:
        df[col] = df[col].astype("string")

print("Rows removed due to missing/blank title:", rows_before_missing_rules - len(df))
print("Rows retained:", len(df))
if COL["transcript"]:
    print("Missing transcripts retained:", int(df[COL["transcript"]].isna().sum()))

Rows removed due to missing/blank title: 0
Rows retained: 2550
Missing transcripts retained: 86


## 8. Parse tags safely

TED tags may be stored as Python-list-like strings. We parse them into a list, standardize whitespace/case, remove blank tag entries, and preserve the original raw tag column.

In [8]:
def parse_tags(value):
    if value is None or pd.isna(value):
        return []
    if isinstance(value, list):
        items = value
    else:
        text_value = str(value).strip()
        if not text_value:
            return []
        try:
            parsed = ast.literal_eval(text_value)
            items = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            # Fallback for simple comma-separated values
            items = text_value.split(",")

    cleaned = []
    seen = set()
    for item in items:
        tag = re.sub(r"\s+", " ", str(item)).strip().lower()
        if tag and tag not in seen:
            cleaned.append(tag)
            seen.add(tag)
    return cleaned

tags_col = COL["tags"]
if tags_col:
    df["tags_list"] = df[tags_col].apply(parse_tags)
else:
    df["tags_list"] = [[] for _ in range(len(df))]

df["tag_count"] = df["tags_list"].apply(len)
df["tags_text"] = df["tags_list"].apply(lambda values: " ".join(values))

display(df[["tags_list", "tag_count", "tags_text"]].head())

,tags_list,tag_count,tags_text
0,"[children, creativity, culture, dance, education, parenting, teaching]",7,children creativity culture dance education parenting teaching
1,"[alternative energy, cars, climate change, culture, environment, global issues, science, sustainability, technology]",9,alternative energy cars climate change culture environment global issues science sustainability technology
2,"[computers, entertainment, interface design, media, music, performance, simplicity, software, technology]",9,computers entertainment interface design media music performance simplicity software technology
3,"[macarthur grant, activism, business, cities, environment, green, inequality, politics, pollution]",9,macarthur grant activism business cities environment green inequality politics pollution
4,"[africa, asia, google, demo, economics, global development, global issues, health, math, statistics, visualizations]",11,africa asia google demo economics global development global issues health math statistics visualizations


## 9. Clean text while preserving the original columns

Text normalization:
1. Decode HTML entities and remove HTML tags.
2. Apply Unicode normalization.
3. Convert text to lowercase.
4. Replace URLs with whitespace.
5. Remove punctuation and non-alphanumeric symbols.
6. Normalize repeated whitespace.

We intentionally do not remove stopwords here. Words such as `not`, `no`, and `never` can change meaning. Stopword filtering and token choices can be tested later during feature engineering.

In [9]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    text_value = html.unescape(str(value))
    text_value = unicodedata.normalize("NFKC", text_value)
    text_value = re.sub(r"<[^>]+>", " ", text_value)
    text_value = re.sub(r"https?://\S+|www\.\S+", " ", text_value)
    text_value = text_value.lower()
    text_value = re.sub(r"[^a-z0-9\s']", " ", text_value)
    text_value = re.sub(r"\s+", " ", text_value).strip()
    return text_value

text_source_map = {
    "title": COL["title"],
    "description": COL["description"],
    "transcript": COL["transcript"],
    "speaker": COL["speaker"],
}

for output_name, source_col in text_source_map.items():
    if source_col and source_col in df.columns:
        df[f"{output_name}_clean"] = df[source_col].apply(clean_text)

display_cols = [c for c in ["title", "title_clean", "description_clean", "transcript_clean"] if c in df.columns]
display(df[display_cols].head(3))

,title,title_clean,description_clean,transcript_clean
0,Do schools kill creativity?,do schools kill creativity,sir ken robinson makes an entertaining and profoundly moving case for creating an education system that nurtures rat...,good morning how are you laughter it's been great hasn't it i've been blown away by the whole thing in fact i'm leav...
1,Averting the climate crisis,averting the climate crisis,with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can ...,thank you so much chris and it's truly a great honor to have the opportunity to come to this stage twice i'm extreme...
2,Simplicity sells,simplicity sells,new york times columnist david pogue takes aim at technology s worst interface design offenders and provides encoura...,music the sound of silence simon garfunkel hello voice mail my old friend laughter i've called for tech support agai...


## 10. Prepare categorical and date fields

- Speaker and occupation are retained as categorical text fields.
- Tags are represented as a list and a joined text field. Multi-hot encoding can be performed later after the final feature vocabulary is decided.
- Publication date is parsed into a separate datetime feature where possible; original value is retained.

In [10]:
if COL["published_date"]:
    source_date = df[COL["published_date"]]
    # TED archive datasets commonly store this field as Unix epoch seconds.
    numeric_date = pd.to_numeric(source_date, errors="coerce")
    parsed_epoch = pd.to_datetime(numeric_date, unit="s", errors="coerce", utc=True)
    parsed_general = pd.to_datetime(source_date, errors="coerce", utc=True)
    df["published_datetime"] = parsed_epoch.where(numeric_date.notna(), parsed_general)
    df["published_year"] = df["published_datetime"].dt.year.astype("Int64")
    df["published_month"] = df["published_datetime"].dt.month.astype("Int64")

for key in ["duration", "views", "comments"]:
    col = COL[key]
    if col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if COL["duration"]:
    df["duration_minutes"] = (df[COL["duration"]] / 60).round(2)

## 11. Create basic numerical content features

These features are useful for data validation and later recommendation work. They do not replace TF-IDF or semantic embeddings, which belong in Module 3.

In [11]:
def word_count(value):
    if value is None or pd.isna(value):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(value)))

for col in ["title_clean", "description_clean", "transcript_clean"]:
    if col in df.columns:
        prefix = col.replace("_clean", "")
        df[f"{prefix}_word_count"] = df[col].apply(word_count)
        df[f"{prefix}_char_count"] = df[col].str.len()

# Content completeness indicators help measure recommendation coverage.
available_text_cols = [c for c in ["title_clean", "description_clean", "transcript_clean", "tags_text"]
                       if c in df.columns]
df["has_transcript"] = df["transcript_clean"].ne("") if "transcript_clean" in df.columns else False
df["has_description"] = df["description_clean"].ne("") if "description_clean" in df.columns else False
df["has_tags"] = df["tags_text"].ne("")
df["content_field_count"] = sum(df[c].ne("").astype(int) for c in available_text_cols)

display(df[[c for c in [
    "title_word_count", "description_word_count", "transcript_word_count",
    "tag_count", "duration_minutes", "has_transcript", "content_field_count"
] if c in df.columns]].describe().T)

,count,mean,std,min,25%,50%,75%,max
title_word_count,2550.0,6.246667,2.207182,1.0,5.0,6.0,8.0,15.0
description_word_count,2550.0,51.966667,18.145765,9.0,39.0,49.0,63.0,129.0
transcript_word_count,2550.0,1997.671765,1018.898397,0.0,1254.25,2006.0,2711.75,9176.0
tag_count,2550.0,7.511373,4.309849,1.0,5.0,6.0,9.0,32.0
duration_minutes,2550.0,13.775153,6.233528,2.25,9.62,14.13,17.445,87.6
content_field_count,2550.0,3.966275,0.180557,3.0,4.0,4.0,4.0,4.0


## 12. Create a unified content field for Module 3

The content field combines available talk information. Missing transcript text does not remove a talk: title, description, speaker, and tags can still contribute.

In [12]:
content_parts = []
for col in ["title_clean", "description_clean", "speaker_clean", "tags_text", "transcript_clean"]:
    if col in df.columns:
        content_parts.append(df[col].fillna(""))

if content_parts:
    combined = content_parts[0].copy()
    for part in content_parts[1:]:
        combined = combined + " " + part
    df["content_text"] = combined.str.replace(r"\s+", " ", regex=True).str.strip()
else:
    df["content_text"] = ""

df["content_word_count"] = df["content_text"].apply(word_count)
display(df[["content_text", "content_word_count"]].head(3))

,content_text,content_word_count
0,do schools kill creativity sir ken robinson makes an entertaining and profoundly moving case for creating an educati...,3197
1,averting the climate crisis with the same humor and humanity he exuded in an inconvenient truth al gore spells out 1...,2212
2,simplicity sells new york times columnist david pogue takes aim at technology s worst interface design offenders and...,3396


## 13. Collaborative-filtering readiness check

Do not create fake user IDs, ratings, or watch histories. Aggregate views, comments, and crowd rating summaries are not equivalent to individual user–talk interactions.

In [13]:
interaction_candidates = [
    c for c in df.columns
    if any(term in c.lower() for term in [
        "user_id", "viewer_id", "watch_history", "user_rating",
        "user_talk", "click_event", "liked_by_user", "saved_by_user"
    ])
]

print("Potential individual interaction columns:", interaction_candidates or "None detected")
if not interaction_candidates:
    print(
        "Conclusion: this dataset alone does not provide a genuine user-item interaction matrix. "
        "Proceed with content-based recommendation first. Collaborative filtering needs "
        "real user_id + talk_id + interaction/rating records."
    )

Potential individual interaction columns: None detected
Conclusion: this dataset alone does not provide a genuine user-item interaction matrix. Proceed with content-based recommendation first. Collaborative filtering needs real user_id + talk_id + interaction/rating records.


## 14. Validate the processed dataset

Checks:
- Talk title is present.
- Talk IDs and URLs are unique when those fields are available.
- Cleaned text fields contain strings.
- Tags are lists.
- Numeric features are numeric.
- Missing transcripts are retained and explicitly marked.

In [14]:
validation_results = []

def add_check(name, passed, detail):
    validation_results.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail)
    })

add_check(
    "Title is present",
    df[title_col].notna().all() and df[title_col].astype("string").str.strip().ne("").all(),
    f"{int(df[title_col].isna().sum())} missing titles"
)

if COL["id"]:
    add_check(
        "Talk ID uniqueness",
        df[COL["id"]].dropna().is_unique,
        f"{int(df[COL['id']].duplicated().sum())} duplicate non-null IDs"
    )

if COL["url"]:
    add_check(
        "URL uniqueness",
        df[COL["url"]].dropna().astype("string").str.strip().is_unique,
        f"{int(df[COL['url']].dropna().astype('string').str.strip().duplicated().sum())} duplicate non-null URLs"
    )

add_check("Tags are parsed lists", df["tags_list"].apply(lambda x: isinstance(x, list)).all(),
          "tags_list contains Python lists")
for col in ["title_clean", "description_clean", "transcript_clean", "content_text"]:
    if col in df.columns:
        add_check(f"{col} is text", df[col].map(lambda x: isinstance(x, str)).all(),
                  "All rows contain strings")

validation_df = pd.DataFrame(validation_results)
display(validation_df)

,check,passed,detail
0,Title is present,True,0 missing titles
1,Talk ID uniqueness,True,0 duplicate non-null IDs
2,URL uniqueness,True,0 duplicate non-null URLs
3,Tags are parsed lists,True,tags_list contains Python lists
4,title_clean is text,True,All rows contain strings
5,description_clean is text,True,All rows contain strings
6,transcript_clean is text,True,All rows contain strings
7,content_text is text,True,All rows contain strings


## 15. Review before exporting

Review the final row count, column count, missingness, and a few representative records. If a validation check fails, investigate before saving.

In [15]:
print("Processed shape:", df.shape)
print("Original raw shape:", raw_df.shape)
print("Rows removed overall:", len(raw_df) - len(df))
print("Missing transcript records retained:",
      int(df[COL["transcript"]].isna().sum()) if COL["transcript"] else "Transcript column unavailable")

display(df.head(5))
display(missing_profile(df).head(20))

Processed shape: (2550, 43)
Original raw shape: (2550, 20)
Rows removed overall: 0
Missing transcript records retained: 86


,talk_id,title,main_speaker,speaker_occupation,description,tags,transcript,related_talks,ratings,views,comments,duration,published_date,event,url,source,film_date,languages,name,num_speaker,tags_list,tag_count,tags_text,title_clean,description_clean,transcript_clean,speaker_clean,published_datetime,published_year,published_month,duration_minutes,title_word_count,title_char_count,description_word_count,description_char_count,transcript_word_count,transcript_char_count,has_transcript,has_description,has_tags,content_field_count,content_text,content_word_count
0,1,Do schools kill creativity?,Ken Robinson,Author/educator,Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (ra...,"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']","Good morning. How are you?(Laughter)It's been great, hasn't it? I've been blown away by the whole thing. In fact, I'...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/images/ted/172559_800x600.jpg', 'speaker': 'Ken Robinson', 'title': 'Bri...","[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, 'name': 'Beautiful', 'count': 4573}, {'id': 9, 'name': 'Ingen...",47227110,4553,1164,1151367060,TED2006,https://www.ted.com/talks/ken_robinson_says_schools_kill_creativity,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,60,Ken Robinson: Do schools kill creativity?,1,"[children, creativity, culture, dance, education, parenting, teaching]",7,children creativity culture dance education parenting teaching,do schools kill creativity,sir ken robinson makes an entertaining and profoundly moving case for creating an education system that nurtures rat...,good morning how are you laughter it's been great hasn't it i've been blown away by the whole thing in fact i'm leav...,ken robinson,2006-06-27 00:11:00+00:00,2006,6,19.40,4,26,21,146,3163,16800,True,True,True,4,do schools kill creativity sir ken robinson makes an entertaining and profoundly moving case for creating an educati...,3197
1,2,Averting the climate crisis,Al Gore,Climate advocate,"With the same humor and humanity he exuded in ""An Inconvenient Truth,"" Al Gore spells out 15 ways that individuals c...","['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainabili...","Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extr...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/images/ted/566c14767bd62c5ff760e483c5b16cd2753328cd_2880x1620.jpg', 'spe...","[{'id': 7, 'name': 'Funny', 'count': 544}, {'id': 3, 'name': 'Courageous', 'count': 139}, {'id': 2, 'name': 'Confusi...",3200520,265,977,1151367060,TED2006,https://www.ted.com/talks/al_gore_on_averting_climate_crisis,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,43,Al Gore: Averting the climate crisis,1,"[alternative energy, cars, climate change, culture, environment, global issues, science, sustainability, technology]",9,alternative energy cars climate change culture environment global issues science sustainability technology,averting the climate crisis,with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can ...,thank you so much chris and it's truly a great honor to have the opportunity to come to this stage twice i'm extreme...,al gore,2006-06-27 00:11:00+00:00,2006,6,16.28,4,27,39,227,2155,11461,True,True,True,4,averting the climate crisis with the same humor and humanity he exuded in an inconvenient truth al gore spells out 1...,2212
2,3,Simplicity sells,David Pogue,Technology columnist,"New York Times columnist David Pogue takes aim at technology’s worst interface-design offenders, and provides encour...","['computers', 'entertainment', 'interface design', 'media', 'music', 'performance', 'simplicity', 'software', 'techn...","(Music: ""The Sound of Silence,"" Simon & Garfunkel)Hello voice mail, my old friend.(Laughter)I've called for tech sup...

,column,null_count,empty_string_count,missing_or_empty,missing_percent
6,transcript,86,0,86,3.37
1,title,0,0,0,0.00
0,talk_id,0,0,0,0.00
2,main_speaker,0,0,0,0.00
3,speaker_occupation,0,0,0,0.00
4,description,0,0,0,0.00
5,tags,0,0,0,0.00
7,related_talks,0,0,0,0.00
8,ratings,0,0,0,0.00
9,views,0,0,0,0.00


## 16. Export processed dataset and preprocessing reports

The raw file is never overwritten. The output is saved as `ted_talks_preprocessed.csv`. This separate filename avoids overwriting any older `ted_talks_clean.csv` until the new pipeline is verified.

In [16]:
if not validation_df["passed"].all():
    print("WARNING: At least one validation check failed. Review the table before using this dataset.")
else:
    print("All validation checks passed.")

# Export list-valued tags as JSON strings so they can be read back reliably from CSV.
export_df = df.copy()
export_df["tags_list"] = export_df["tags_list"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)

export_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

missing_after = missing_profile(df)
missing_after.to_csv(STATS_DIR / "preprocessing_missing_values_after.csv", index=False)
validation_df.to_csv(STATS_DIR / "preprocessing_validation.csv", index=False)

summary = {
    "input_file": str(RAW_PATH),
    "output_file": str(OUTPUT_PATH),
    "input_rows": int(len(raw_df)),
    "output_rows": int(len(df)),
    "input_columns": int(raw_df.shape[1]),
    "output_columns": int(df.shape[1]),
    "exact_duplicate_rows_removed": exact_duplicates_before,
    "duplicate_talk_ids_removed": duplicate_id_removed,
    "rows_removed_for_missing_title": int(rows_before_missing_rules - (len(df) + duplicate_id_removed)),
    "missing_transcripts_retained": int(df[COL["transcript"]].isna().sum()) if COL["transcript"] else None,
    "validation_passed": bool(validation_df["passed"].all())
}

summary_path = REPORT_DIR / "preprocessing_summary.json"
summary_path.write_text(json.dumps(summary, indent=4), encoding="utf-8")
print(json.dumps(summary, indent=4))
print("\nSaved processed dataset:", OUTPUT_PATH)

All validation checks passed.
{
    "input_file": "c:\\Users\\DELL\\Desktop\\INTERNSHIPS\\Infosys_SpringBoard\\Ted_Talk_Recommendation\\Scraped\\Data\\Raw\\ted_talks.csv",
    "output_file": "c:\\Users\\DELL\\Desktop\\INTERNSHIPS\\Infosys_SpringBoard\\Ted_Talk_Recommendation\\Scraped\\Data\\Processed\\ted_talks_preprocessed.csv",
    "input_rows": 2550,
    "output_rows": 2550,
    "input_columns": 20,
    "output_columns": 43,
    "exact_duplicate_rows_removed": 0,
    "duplicate_talk_ids_removed": 0,
    "rows_removed_for_missing_title": 0,
    "missing_transcripts_retained": 86,
    "validation_passed": true
}

Saved processed dataset: c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Scraped\Data\Processed\ted_talks_preprocessed.csv


## 17. Reload test — confirm the exported file is readable

A saved CSV should be readable again. List-valued tags are JSON strings after export and can be reconstructed in a later notebook with `json.loads`.

In [17]:
check_df = pd.read_csv(OUTPUT_PATH, low_memory=False)
print("Reloaded processed shape:", check_df.shape)
print("Output exists:", OUTPUT_PATH.exists())
print("Output size (MB):", round(OUTPUT_PATH.stat().st_size / (1024 * 1024), 2))
display(check_df.head(3))

Reloaded processed shape: (2550, 43)
Output exists: True
Output size (MB): 89.14


,talk_id,title,main_speaker,speaker_occupation,description,tags,transcript,related_talks,ratings,views,comments,duration,published_date,event,url,source,film_date,languages,name,num_speaker,tags_list,tag_count,tags_text,title_clean,description_clean,transcript_clean,speaker_clean,published_datetime,published_year,published_month,duration_minutes,title_word_count,title_char_count,description_word_count,description_char_count,transcript_word_count,transcript_char_count,has_transcript,has_description,has_tags,content_field_count,content_text,content_word_count
0,1,Do schools kill creativity?,Ken Robinson,Author/educator,Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (ra...,"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']","Good morning. How are you?(Laughter)It's been great, hasn't it? I've been blown away by the whole thing. In fact, I'...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/images/ted/172559_800x600.jpg', 'speaker': 'Ken Robinson', 'title': 'Bri...","[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, 'name': 'Beautiful', 'count': 4573}, {'id': 9, 'name': 'Ingen...",47227110,4553,1164,1151367060,TED2006,https://www.ted.com/talks/ken_robinson_says_schools_kill_creativity,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,60,Ken Robinson: Do schools kill creativity?,1,"[""children"", ""creativity"", ""culture"", ""dance"", ""education"", ""parenting"", ""teaching""]",7,children creativity culture dance education parenting teaching,do schools kill creativity,sir ken robinson makes an entertaining and profoundly moving case for creating an education system that nurtures rat...,good morning how are you laughter it's been great hasn't it i've been blown away by the whole thing in fact i'm leav...,ken robinson,2006-06-27 00:11:00+00:00,2006,6,19.40,4,26,21,146,3163,16800,True,True,True,4,do schools kill creativity sir ken robinson makes an entertaining and profoundly moving case for creating an educati...,3197
1,2,Averting the climate crisis,Al Gore,Climate advocate,"With the same humor and humanity he exuded in ""An Inconvenient Truth,"" Al Gore spells out 15 ways that individuals c...","['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainabili...","Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extr...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/images/ted/566c14767bd62c5ff760e483c5b16cd2753328cd_2880x1620.jpg', 'spe...","[{'id': 7, 'name': 'Funny', 'count': 544}, {'id': 3, 'name': 'Courageous', 'count': 139}, {'id': 2, 'name': 'Confusi...",3200520,265,977,1151367060,TED2006,https://www.ted.com/talks/al_gore_on_averting_climate_crisis,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,43,Al Gore: Averting the climate crisis,1,"[""alternative energy"", ""cars"", ""climate change"", ""culture"", ""environment"", ""global issues"", ""science"", ""sustainabili...",9,alternative energy cars climate change culture environment global issues science sustainability technology,averting the climate crisis,with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can ...,thank you so much chris and it's truly a great honor to have the opportunity to come to this stage twice i'm extreme...,al gore,2006-06-27 00:11:00+00:00,2006,6,16.28,4,27,39,227,2155,11461,True,True,True,4,averting the climate crisis with the same humor and humanity he exuded in an inconvenient truth al gore spells out 1...,2212
2,3,Simplicity sells,David Pogue,Technology columnist,"New York Times columnist David Pogue takes aim at technology’s worst interface-design offenders, and provides encour...","['computers', 'entertainment', 'interface design', 'media', 'music', 'performance', 'simplicity', 'software', 'techn...","(Music: ""The Sound of Silence,"" Simon & Garfunkel)Hello voice mail, my ol

## 18. Module 2 preprocessing conclusion

After successful execution, the output dataset is ready for Module 3 feature engineering.

### What this notebook produces
- `Scraped/Data/Processed/ted_talks_preprocessed.csv`
- `outputs/reports/preprocessing_summary.json`
- `outputs/statistics/preprocessing_missing_values_after.csv`
- `outputs/statistics/preprocessing_validation.csv`

### Module 3 hand-off
Use `content_text` or the individual cleaned text fields as input for TF-IDF / other vectorization experiments. Keep `talk_id` as the stable item identifier. Compare title/description/tags-only content against transcript-inclusive content if useful.

### Important interpretation
- Missing transcript does not mean a talk is unusable.
- Keep original source columns for traceability.
- Do not use aggregate views as individual user preferences.
- Do not train or evaluate a recommendation model until a suitable evaluation strategy and data split are defined.

In [18]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nProcessed dataset exists:")
print(OUTPUT_PATH.exists())

print("\nProcessed dataset location:")
print(OUTPUT_PATH)

print("\nFiles in processed folder:")
for file in PROCESSED_DIR.glob("*"):
    print(file.name)

print("\nFiles in output folder:")
for file in OUTPUT_DIR.rglob("*"):
    if file.is_file():
        print(file)

Current working directory:
c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\notebooks

Project root:
c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation

Processed dataset exists:
True

Processed dataset location:
c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Scraped\Data\Processed\ted_talks_preprocessed.csv

Files in processed folder:
ted_talks_clean.csv
ted_talks_preprocessed.csv

Files in output folder:
c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_2_Data_Exploration_Preprocessing\outputs\reports\data_quality_report.txt
c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_2_Data_Exploration_Preprocessing\outputs\reports\preprocessing_summary.json
c:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_2_Data_Exploration_Preprocessing\outputs\statistics\missing_value_summary.csv
c:\Users\DELL\Desktop